# 🧨 Cookbook: refusals, and the branchless patterns behind them

Part one: FAFO — every operation that would *reveal* a value answers with a
teaching error. Part two: the rewrite patterns that compute the same
decisions **without** revealing anything, the way constant-time crypto code
does it.

In [ ]:
%%time
%pip install -q git+https://github.com/PDPG-lab/pypdpg

In [ ]:
import numpy as np
import pypdpg as pdpg

ctx = pdpg.Context.create()
ctx.save_public("processor.ctx")
x = np.random.default_rng(0).uniform(-3, 3, 200)
pdpg.encrypt(x, ctx).save("data.enc")

# the processor's seat: evaluation keys only
pdpg.activate("processor.ctx")
pdpg.install()
X = np.load("data.enc")

def fafo(attempt):
    try:
        attempt()
        print("!? that unexpectedly worked")
    except pdpg.EncryptedOperationError as e:
        print(f"⛔ {e}")

In [ ]:
fafo(lambda: X > 0)            # who passed? not your call to make

In [ ]:
fafo(lambda: np.exp(X))        # transcendental functions don't exist here

In [ ]:
fafo(lambda: X / X)            # no ciphertext division

In [ ]:
fafo(lambda: np.sort(X))       # sorting means comparing means reading

In [ ]:
fafo(lambda: bool(X))          # not even a yes/no leaks

In [ ]:
fafo(lambda: np.asarray(X))    # numpy can't materialize the plaintext either

In [ ]:
# 🫵 your turn — raw, no safety net; errors here are the point
# X ** 0.5, X[0], abs(X), np.argmax(X), X == X ...

---
# Part two: the rewrite

Data-dependent **control flow** needs a plaintext boolean — never available
here. Data-dependent **selection** is just arithmetic:

```
if x > 0: b else c    →    gate·b + (1−gate)·c
```

with `gate = sigmoid(x)`, an encrypted soft indicator in [0, 1].

In [ ]:
b = np.full(200, +10.0)     # the "approve" branch
c = np.full(200, -10.0)     # the "review" branch

gate = pdpg.approx.sigmoid(X)          # encrypted indicator, depth 2
result = gate * b + (1 - gate) * c     # both branches paid, selection by math
result

In [ ]:
# the processor computed the split blind; the check belongs to the key holder
result.save("result.enc")

decrypted = pdpg.load("result.enc", ctx).decrypt()   # ctx = the controller's full context
plain_gate = pdpg.approx.sigmoid(x)
expected = plain_gate * b + (1 - plain_gate) * c
print("max abs err vs plain:", np.abs(decrypted - expected).max())
print("branches correct:", np.all((decrypted > 0) == (x > 0)))

## The rules (constant-time programmers already know them)

- **Both branches always execute.** An early exit would leak which branch ran.
- **The gate is soft** under CKKS — a gray zone near the threshold. Exact 0/1
  gates arrive with backend-side comparison circuits (TFHE-class); the pattern won't change.
- **Loops need fixed worst-case bounds.** A data-dependent trip count is
  control flow.
- **Shapes can't depend on data.** No filter that returns fewer rows — the
  row *count* would leak. Mask to full shape instead.

<sub>A [PDPG-lab](https://pdpglab.xyz) project. Current backend:
[TenSEAL](https://github.com/OpenMined/TenSEAL) (CKKS). More recipes in
[demo/cookbook](https://github.com/PDPG-lab/pypdpg/tree/main/demo/cookbook).</sub>